# Phân tích khảo sát hành vi đánh giá du lịch

Dữ liệu đầu vào là `data/SURVEY/tourist_survey.csv`, được xuất từ biểu mẫu khảo sát có sự đồng ý của người trả lời.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent
data_dir = project_root / 'data'
analysis_dir = data_dir / 'analysis'
analysis_dir.mkdir(exist_ok=True)
survey_file = data_dir / 'SURVEY' / 'tourist_survey.csv'


In [ ]:
required_columns = ['respondent_id', 'destination', 'overall_rating', 'landscape_rating',
                    'service_rating', 'price_rating', 'food_rating', 'transport_rating',
                    'safety_rating', 'cleanliness_rating', 'revisit_intention',
                    'recommend_intention', 'review_text']
survey = pd.read_csv(survey_file)
missing_columns = set(required_columns) - set(survey.columns)
if missing_columns:
    raise ValueError(f'Tệp khảo sát thiếu cột: {sorted(missing_columns)}')
print(f'Số phản hồi hiện có: {len(survey)}')
survey.head()

In [ ]:
rating_columns = ['overall_rating', 'landscape_rating', 'service_rating', 'price_rating',
                  'food_rating', 'transport_rating', 'safety_rating', 'cleanliness_rating']
positive_words = ['đẹp', 'tốt', 'hài lòng', 'thân thiện', 'ngon', 'sạch', 'tuyệt', 'an toàn']
negative_words = ['tệ', 'xấu', 'đắt', 'bẩn', 'kém', 'chậm', 'lừa', 'nguy hiểm']

def sentiment_score(text):
    text = str(text).lower()
    return sum(word in text for word in positive_words) - sum(word in text for word in negative_words)

clean = survey.copy()
for column in rating_columns:
    clean[column] = pd.to_numeric(clean[column], errors='coerce')
clean['revisit_binary'] = clean['revisit_intention'].astype(str).str.lower().map({'có': 1, 'co': 1, 'yes': 1, '1': 1, 'không': 0, 'khong': 0, 'no': 0, '0': 0})
clean['recommend_binary'] = clean['recommend_intention'].astype(str).str.lower().map({'có': 1, 'co': 1, 'yes': 1, '1': 1, 'không': 0, 'khong': 0, 'no': 0, '0': 0})
clean['sentiment_score'] = clean['review_text'].fillna('').apply(sentiment_score)
clean = clean.dropna(subset=['destination', 'overall_rating'])
clean.to_csv(analysis_dir / 'tourist_survey_clean.csv', encoding='utf-8-sig', index=False)
clean.head()

In [ ]:
if clean.empty:
    print('Chưa có phản hồi để phân tích. Hãy nhập dữ liệu từ biểu mẫu khảo sát.')
else:
    destination_rating_stats = (clean.groupby('destination', as_index=False)
        .agg(respondent_count=('respondent_id', 'count'),
             average_overall_rating=('overall_rating', 'mean'),
             revisit_rate=('revisit_binary', 'mean'),
             recommend_rate=('recommend_binary', 'mean'),
             average_sentiment=('sentiment_score', 'mean'))
        .sort_values('average_overall_rating', ascending=False))
    destination_rating_stats.to_csv(analysis_dir / 'destination_rating_stats.csv', encoding='utf-8-sig', index=False)
    display(destination_rating_stats)


In [ ]:
factor_columns = ['landscape_rating', 'service_rating', 'price_rating', 'food_rating',
                  'transport_rating', 'safety_rating', 'cleanliness_rating']
model_data = clean.dropna(subset=factor_columns + ['revisit_binary'])
if len(model_data) < 30 or model_data['revisit_binary'].nunique() < 2:
    print('Cần ít nhất 30 phản hồi hợp lệ và cả hai nhóm Có/Không để mô hình hóa ý định quay lại.')
else:
    scaler = StandardScaler()
    X = scaler.fit_transform(model_data[factor_columns])
    y = model_data['revisit_binary']
    model = LogisticRegression(max_iter=1000, random_state=365).fit(X, y)
    coefficients = pd.DataFrame({'factor': factor_columns, 'coefficient': model.coef_[0]}).sort_values('coefficient', ascending=False)
    coefficients.to_csv(analysis_dir / 'revisit_factor_coefficients.csv', encoding='utf-8-sig', index=False)
    display(coefficients)
